In [94]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,accuracy_score,f1_score,confusion_matrix
from imblearn.over_sampling import SMOTE
import xgboost as xgb

#Cyclone Dataset

In [49]:
cyc = pd.read_csv('Cyclone.csv', low_memory=False, skiprows=[1])

In [50]:
cyc.shape

(62605, 174)

In [51]:
cyc.head().T

,0,1,2,3,4
SID,1842298N11080,1842298N11080,1842298N11080,1842298N11080,1842298N11080
SEASON,1842,1842,1842,1842,1842
NUMBER,1,1,1,1,1
BASIN,NI,NI,NI,NI,NI
SUBBASIN,BB,BB,BB,BB,BB
...,...,...,...,...,...
USA_SEARAD_SE,,,,,
USA_SEARAD_SW,,,,,
USA_SEARAD_NW,,,,,
STORM_SPEED,9,9,9,9,9


In [52]:
cyc.describe()

,SEASON,NUMBER,LAT,LON,DIST2LAND,USA_SSHS
count,62605.000000,62605.000000,62605.000000,62605.000000,62605.000000,62605.000000
mean,1952.935596,57.101030,17.622706,82.998516,216.912499,-3.784314
std,40.916771,31.309144,6.180089,17.828134,270.053131,2.164652
min,1842.000000,1.000000,0.700000,-87.700000,0.000000,-5.000000
25%,1922.000000,31.000000,13.000000,77.900000,0.000000,-5.000000
50%,1957.000000,51.000000,18.300000,84.700000,112.000000,-5.000000
75%,1986.000000,84.000000,21.900000,88.600000,361.000000,-3.000000
max,2025.000000,140.000000,83.000000,163.700000,2271.000000,5.000000


In [53]:
cyc['SEASON'] = pd.to_numeric(cyc['SEASON'], errors='coerce')
cyc['WMO_WIND'] = pd.to_numeric(cyc['WMO_WIND'], errors='coerce')
cyc['USA_WIND'] = pd.to_numeric(cyc['USA_WIND'], errors='coerce')
cyc['LAT'] = pd.to_numeric(cyc['LAT'], errors='coerce')
cyc['LON'] = pd.to_numeric(cyc['LON'], errors='coerce')
cyc['ISO_TIME'] = pd.to_datetime(cyc['ISO_TIME'], errors='coerce')

In [54]:
cyc = cyc[(cyc['SUBBASIN'].isin(['BB', 'AS'])) & (cyc['SEASON'] >= 1990)].copy()
print('Track points after filtering:', cyc.shape[0])
print('Unique storms:', cyc['SID'].nunique())

Track points after filtering: 12452
Unique storms: 356


In [55]:
cyc['intensity_for_selection'] = cyc['WMO_WIND'].fillna(cyc['USA_WIND'])

In [56]:
def pick_peak(g):
    if g['intensity_for_selection'].notna().any():
        return g.loc[g['intensity_for_selection'].idxmax()]
    return g.iloc[len(g) // 2]

In [57]:
cyc_storms = cyc.groupby('SID', group_keys=False).apply(pick_peak)

cyc_events = pd.DataFrame({
    'latitude': cyc_storms['LAT'],
    'longitude': cyc_storms['LON'],
    'month': cyc_storms['ISO_TIME'].dt.month,
    'year': cyc_storms['SEASON'],
    'label': 'cyclone'
}).reset_index(drop=True)

/tmp/ipykernel_2192/4027165373.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cyc_storms = cyc.groupby('SID', group_keys=False).apply(pick_peak)


In [58]:
print('Cyclone events (one per storm):', cyc_events.shape)
cyc_events.head()

Cyclone events (one per storm): (356, 5)


,latitude,longitude,month,year,label
0,10.5,86.4,4,1990,cyclone
1,13.1,81.8,5,1990,cyclone
2,22.0,87.0,6,1990,cyclone
3,19.0,87.5,8,1990,cyclone
4,20.0,88.5,8,1990,cyclone


#EarthQuake Dataset

In [59]:
eq = pd.read_csv('query.csv')

In [60]:
eq = eq[eq['type'] == 'earthquake'].copy()
eq['time'] = pd.to_datetime(eq['time'], errors='coerce')

In [61]:
eq_events = pd.DataFrame({
    'latitude': eq['latitude'],
    'longitude': eq['longitude'],
    'month': eq['time'].dt.month,
    'year': eq['time'].dt.year,
    'label': 'earthquake'
}).reset_index(drop=True)

In [62]:
print('Earthquake events:', eq_events.shape)
eq_events.head()

Earthquake events: (15576, 5)


,latitude,longitude,month,year,label
0,26.0392,97.0716,12,2025,earthquake
1,35.7405,88.5284,12,2025,earthquake
2,26.7466,92.3604,12,2025,earthquake
3,23.3943,82.5615,12,2025,earthquake
4,23.7096,70.4196,12,2025,earthquake


#Merge Datasets

In [63]:
data = pd.concat([cyc_events, eq_events], ignore_index=True).dropna()

In [65]:
data.shape

(15932, 5)

In [64]:
data.head()

,latitude,longitude,month,year,label
0,10.5,86.4,4,1990,cyclone
1,13.1,81.8,5,1990,cyclone
2,22.0,87.0,6,1990,cyclone
3,19.0,87.5,8,1990,cyclone
4,20.0,88.5,8,1990,cyclone


In [66]:
data['label'].value_counts()

,count
label,
earthquake,15576
cyclone,356


In [67]:
# print('Combined dataset:', data.shape)
# print(data['label'].value_counts())
# print()
# print('Year range check -> cyclone:', cyc_events['year'].min(), '-', cyc_events['year'].max(),
#       '| earthquake:', eq_events['year'].min(), '-', eq_events['year'].max())

#Feature Engineering

In [68]:
data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12)
data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)

In [87]:
# 1 = cyclone, 0 = earthquake
X = data[['latitude', 'longitude', 'month_sin', 'month_cos']]
y = (data['label'] == 'cyclone').astype(int)

#Training Model

In [77]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [78]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

In [79]:
model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss'
)

In [80]:
model.fit(X_train, y_train)
pred = model.predict(X_test)

In [81]:
print(classification_report(y_test, pred, target_names=['earthquake', 'cyclone']))
print('Macro F1:', f1_score(y_test, pred, average='macro'))

              precision    recall  f1-score   support

  earthquake       1.00      0.99      1.00      3116
     cyclone       0.79      0.92      0.85        71

    accuracy                           0.99      3187
   macro avg       0.90      0.96      0.92      3187
weighted avg       0.99      0.99      0.99      3187

Macro F1: 0.9229880239080576


#SMOTE

In [84]:
sm = SMOTE(random_state=42)

In [90]:
x_train_sm,y_train_sm = sm.fit_resample(X_train,y_train)

In [86]:
print('After SMOTE, train class balance:', pd.Series(y_train_sm).value_counts().to_dict())

After SMOTE, train class balance: {0: 12460, 1: 12460}


In [88]:
model_smote = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                               random_state=42, eval_metric='logloss')

In [92]:
model_smote.fit(x_train_sm, y_train_sm)
pred_smote =model_smote.predict(X_test)

In [97]:
print(classification_report(y_test, pred_smote, target_names=['earthquake', 'cyclone']))

              precision    recall  f1-score   support

  earthquake       1.00      0.99      1.00      3116
     cyclone       0.75      0.93      0.83        71

    accuracy                           0.99      3187
   macro avg       0.87      0.96      0.91      3187
weighted avg       0.99      0.99      0.99      3187

Macro F1: 0.9129221755035748


In [99]:
print('Macro F1:', f1_score(y_test, pred_smote, average='macro'))

Macro F1: 0.9129221755035748


In [100]:
print('Confusion matrix:')
print(confusion_matrix(y_test, pred_smote))

Confusion matrix:
[[3094   22]
 [   5   66]]


#Feature Importance

In [102]:
for f, imp in zip(X.columns, model.feature_importances_):
    print(f'{f}: {imp:.3f}')

latitude: 0.411
longitude: 0.457
month_sin: 0.061
month_cos: 0.072
